# Stage 07: Ward Master Dataset (Crosswalk-Corrected 5-Election Panel)

**Pipeline Stage:** Data Merging (Group 7 Rebuilt)  
**Primary Input:** `data/processed/01_ward_election_panel/ward_election_panel_2000_2021_crosswalk_corrected.csv`  
**Crosswalk Input:** `data/processed/01_ward_election_panel/voting_district_to_2021_ward_crosswalk.csv`  
**Context Inputs:** `data/processed/02_province_context_panel/`, `data/processed/03_municipality_context/`  
**Output File:** `data/processed/07_ward_master_dataset/ward_master_dataset_2000_2021_crosswalk_corrected.csv`  

### Architectural Rationale & Boundary Correction
In the initial 3-election pipeline (2011-2021), wards were matched naively on numeric ward numbers. However, voting district analysis revealed that ~33% of voting districts were renumbered to different wards between 2011 and 2021 due to municipal demarcation boundary changes. This notebook integrates the crosswalk-corrected panel spanning five local government elections (2000, 2006, 2011, 2016, 2021) anchored to the 2021 reference ward geography, providing true longitudinal continuity for 863 out of 901 wards (95.8%).

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Dynamic base path resolution
BASE_DIR = Path.cwd().parent.parent if Path.cwd().name == 'Data Merging' else Path.cwd()
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print('Project Root:', BASE_DIR)


### Step 1: Load Crosswalk and Corrected Election Panel
We map each ward to its canonical 2021 reference municipality and aggregate voting districts into unique ward-year records.

In [2]:
cw_path = PROCESSED_DIR / '01_ward_election_panel' / 'voting_district_to_2021_ward_crosswalk.csv'
df_cw = pd.read_csv(cw_path)
ward_to_muni = df_cw.groupby('Ward2021Reference')['Municipality2021'].first().to_dict()

panel_path = PROCESSED_DIR / '01_ward_election_panel' / 'ward_election_panel_2000_2021_crosswalk_corrected.csv'
df_panel = pd.read_csv(panel_path)
df_panel['Municipality2021'] = df_panel['Ward'].map(ward_to_muni)

# Ward-year strict aggregation (resolves multi-municipality historical district merges)
spine = df_panel.groupby(['Ward', 'ElectionYear', 'Municipality2021'], as_index=False).agg(
    RegisteredVoters=('RegisteredVoters', 'sum'),
    TotalVotesCast=('TotalVotesCast', 'sum'),
    VotingStationsMatched=('VotingStationsMatched', 'sum')
)
spine['TurnoutRate'] = (spine['TotalVotesCast'] / spine['RegisteredVoters']) * 100
spine['Province'] = 'KwaZulu-Natal'
spine['MunicipalityCode'] = spine['Municipality2021'].str.extract(r'^(KZN\d+|ETH)', expand=False)

print(f'Spine rows: {len(spine):,}, Unique wards: {spine["Ward"].nunique()}')
print('Rows by Election Year:\n', spine['ElectionYear'].value_counts().sort_index())


### Step 2: Merge Municipality Demographics Context
We join demographic indicators from `03_municipality_context` (population, household density, settlement type) via the validated 44-municipality IEC code mapping.

In [3]:
muni_ctx = pd.read_csv(PROCESSED_DIR / '03_municipality_context' / 'municipality_context.csv')
code_map = {
    'ETH': 'Ethekwini Metropolitan Municipality', 'KZN212': 'Umdoni Local Municipality', 'KZN213': 'Umzumbe Local Municipality',
    'KZN214': 'UMuziwabantu Local Municipality', 'KZN216': 'Ray Nkonyeni Local Municipality', 'KZN221': 'uMshwathi Local Municipality',
    'KZN222': 'uMngeni Local Municipality', 'KZN223': 'Mpofana Local Municipality', 'KZN224': 'Impendle Local Municipality',
    'KZN225': 'The Msunduzi Local Municipality', 'KZN226': 'Mkhambathini Local Municipality', 'KZN227': 'Richmond Local Municipality',
    'KZN235': 'Okhahlamba Local Municipality', 'KZN237': 'Inkosi Langalibalele Local Municipality', 'KZN238': 'Alfred Duma Local Municipality',
    'KZN241': 'Endumeni Local Municipality', 'KZN242': 'Nqutu Local Municipality', 'KZN244': 'Msinga Local Municipality',
    'KZN245': 'Umvoti Local Municipality', 'KZN252': 'Newcastle Local Municipality', 'KZN253': 'Emadlangeni Local Municipality',
    'KZN254': 'Dannhauser Local Municipality', 'KZN261': 'eDumbe Local Municipality', 'KZN262': 'UPhongolo Local Municipality',
    'KZN263': 'Abaqulusi Local Municipality', 'KZN265': 'Nongoma Local Municipality', 'KZN266': 'Ulundi Local Municipality',
    'KZN271': 'Umhlabuyalingana Local Municipality', 'KZN272': 'Jozini Local Municipality', 'KZN275': 'Mtubatuba Local Municipality',
    'KZN276': 'Big Five Hlabisa Local Municipality', 'KZN281': 'Mfolozi Local Municipality', 'KZN282': 'uMhlathuze Local Municipality',
    'KZN284': 'uMlalazi Local Municipality', 'KZN285': 'Mthonjaneni Local Municipality', 'KZN286': 'Nkandla Local Municipality',
    'KZN291': 'Mandeni Local Municipality', 'KZN292': 'KwaDukuza Local Municipality', 'KZN293': 'Ndwedwe Local Municipality',
    'KZN294': 'Maphumulo Local Municipality', 'KZN433': 'Greater Kokstad Local Municipality', 'KZN434': 'Ubuhlebezwe Local Municipality',
    'KZN435': 'Umzimkhulu Local Municipality', 'KZN436': 'Dr Nkosazana Dlamini Zuma Local Municipality'
}
spine['CurrentMunicipality'] = spine['MunicipalityCode'].map(code_map)
master_df = spine.merge(muni_ctx, left_on='CurrentMunicipality', right_on='Municipality', how='left')
print('Merged with municipality context. Shape:', master_df.shape)


### Step 3: Merge Provincial Poverty & Socioeconomic Context
We align provincial socioeconomic and poverty metrics from `02_province_context_panel` using the closest available historical timeline.

In [4]:
prov_ctx = pd.read_csv(PROCESSED_DIR / '02_province_context_panel' / 'province_context_panel_2011_2023.csv')
province_year_map = {2000: 2011, 2006: 2011, 2011: 2011, 2016: 2015, 2021: 2023}
master_df['ProvinceContextYear'] = master_df['ElectionYear'].map(province_year_map)

prov_ctx_join = prov_ctx.copy()
prov_ctx_join['geography_name'] = 'KwaZulu-Natal'
master_df = master_df.merge(
    prov_ctx_join,
    left_on=['Province', 'ProvinceContextYear'],
    right_on=['geography_name', 'year'],
    how='left'
)

drop_cols = ['geography_level', 'geography_name', 'year', 'Municipality_y']
master_df = master_df.drop(columns=[c for c in drop_cols if c in master_df.columns])
master_df = master_df.rename(columns={'Municipality_x': 'Municipality'})

out_master_path = PROCESSED_DIR / '07_ward_master_dataset' / 'ward_master_dataset_2000_2021_crosswalk_corrected.csv'
master_df.to_csv(out_master_path, index=False)
print(f'Successfully saved corrected master dataset: {out_master_path.name} ({master_df.shape})')
